# Data pipeline

Downloads two months of Lichess games (November 2020 and June 2024), keeps games where both
players are rated 2200+ with at least 8 minutes on the clock, turns them into training examples,
drops examples where the human move lost more than 50 centipawns according to Stockfish, and
splits them by game into train, validation and test sets. The results are saved to Google Drive.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
if not os.path.isdir("/content/chess-llm-thesis"):
    !git clone -q https://github.com/Lefterisstam/chess-llm-thesis.git /content/chess-llm-thesis
%cd /content/chess-llm-thesis
!pip install -q -r requirements.txt
!apt-get install -qq stockfish

## Download

The two files are 23 GB and 29 GB, so they go to the local disk of the Colab machine.

In [ ]:
RAW_DIR = "/content/lichess_raw"
WORK_DIR = "/content/drive/MyDrive/chess_llm_data"

!python data/scripts/download_lichess.py --months 2020-11 2024-06 --output-dir {RAW_DIR}

## Run the pipeline

The Stockfish step takes several hours. If the session stops, run the cell again: finished steps
are skipped and the validation continues from where it stopped.

In [ ]:
!python data/scripts/run_pipeline.py --raw-dir {RAW_DIR} --work-dir {WORK_DIR}

## Check the output

In [ ]:
import json

for name in ["train", "val", "test"]:
    with open(f"{WORK_DIR}/processed/{name}.jsonl") as f:
        print(name, sum(1 for _ in f))

with open(f"{WORK_DIR}/processed/train.jsonl") as f:
    example = json.loads(f.readline())
print(example["text"])
print(example["metadata"])